# RAG Pipeline — Exploration Notebook
**Meridian Capital Group HR Policy Q&A**

This notebook walks through the full RAG pipeline interactively:
1. Document loading and inspection
2. Chunking strategy and analysis
3. Embedding generation
4. Vector store operations
5. Retrieval comparison (k=1 vs k=5, score thresholds)
6. End-to-end generation
7. Evaluation on ground truth Q&A pairs

> **Before running:** Copy `.env.example` to `.env` and add your API keys.


## 0. Setup

In [ ]:
%pip install -r ../requirements.txt -q

In [ ]:
import os, sys, json
from pathlib import Path
from dotenv import load_dotenv

# Add src to path
sys.path.insert(0, str(Path("..").resolve()))
load_dotenv("../.env")

# Verify keys are loaded
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not set in .env"
print("Environment loaded")
print(f"  LLM provider  : {os.getenv('LLM_PROVIDER', 'openai')}")
print(f"  Embed model   : {os.getenv('EMBED_MODEL', 'text-embedding-3-small')}")
print(f"  Chroma path   : {os.getenv('CHROMA_PERSIST_DIR', 'data/chroma')}")


## 1. Document Loading

The pipeline uses LlamaIndex `SimpleDirectoryReader` which handles PDF parsing
via `pypdf`. We inspect the raw document output before any chunking.


In [ ]:
from src.ingestion.loader import load_documents

docs = load_documents("../data/raw")
print(f"Documents loaded : {len(docs)}")
for doc in docs:
    print(f"  File     : {doc.metadata.get('file_name')}")
    print(f"  Pages    : {doc.metadata.get('page_label')}")
    print(f"  Chars    : {len(doc.text):,}")
    print(f"  Preview  : {doc.text[:200].replace(chr(10), ' ')}...")
    print()


In [ ]:
# Inspect the raw text for a specific section
target = "Section 2"
for doc in docs:
    idx = doc.text.find(target)
    if idx != -1:
        print(f"Found '{target}' at char {idx}")
        print(doc.text[idx:idx+500])
        break


## 2. Chunking Strategy

We use LlamaIndex `SentenceSplitter` with `chunk_size=512` tokens and
`chunk_overlap=50` tokens. The overlap ensures context is preserved across
chunk boundaries — important for multi-sentence policy definitions.

We'll compare three configurations to understand the tradeoffs.


In [ ]:
from src.ingestion.chunker import chunk_documents, get_chunk_metadata

# Default config: 512 tokens, 50 overlap
chunks_default = chunk_documents(docs)
print(f"Default (512/50): {len(chunks_default)} chunks")

# Smaller chunks — more granular retrieval, less context per chunk
chunks_small = chunk_documents(docs, chunk_size=256, chunk_overlap=25)
print(f"Small   (256/25): {len(chunks_small)} chunks")

# Larger chunks — more context per chunk, less precise retrieval
chunks_large = chunk_documents(docs, chunk_size=1024, chunk_overlap=100)
print(f"Large (1024/100): {len(chunks_large)} chunks")


In [ ]:
# Inspect chunk size distribution for default config
metadata = get_chunk_metadata(chunks_default)
char_lengths = [len(m["text_preview"]) for m in metadata]

import statistics
print(f"Chunk count    : {len(metadata)}")
print(f"Avg chars      : {statistics.mean(char_lengths):.0f}")
print(f"Median chars   : {statistics.median(char_lengths):.0f}")
print(f"Min chars      : {min(char_lengths)}")
print(f"Max chars      : {max(char_lengths)}")
print()

# Show the first 3 chunks with metadata
for m in metadata[:3]:
    print(f"[Chunk {m['chunk_index']}]")
    print(f"  File    : {m['file_name']}")
    print(f"  Page    : {m['page_label']}")
    print(f"  Preview : {m['text_preview'][:150]}...")
    print()


In [ ]:
# Visualise chunk length distribution
try:
    import matplotlib.pyplot as plt
    all_lengths = [len(c.get_content()) for c in chunks_default]
    plt.figure(figsize=(9, 4))
    plt.hist(all_lengths, bins=20, color="#1a1a2e", edgecolor="white", alpha=0.85)
    plt.xlabel("Chunk length (characters)")
    plt.ylabel("Count")
    plt.title("Chunk Length Distribution — Default (512 tokens / 50 overlap)")
    plt.tight_layout()
    plt.savefig("chunk_distribution.png", dpi=120)
    plt.show()
    print("Plot saved to chunk_distribution.png")
except ImportError:
    print("matplotlib not installed — skipping plot")
    lengths = [len(c.get_content()) for c in chunks_default]
    buckets = {}
    for l in lengths:
        b = (l // 200) * 200
        buckets[b] = buckets.get(b, 0) + 1
    for b in sorted(buckets):
        print(f"  {b:4d}–{b+199}: {'█' * buckets[b]} ({buckets[b]})")


## 3. Embedding Generation

We use OpenAI `text-embedding-3-small` (1536 dimensions). This section
generates embeddings for a sample of chunks and explores the embedding space.


In [ ]:
from src.embedding.embedder import get_embed_model, embed_texts, embed_query

embed_model = get_embed_model()
print(f"Embed model : {embed_model.model}")


In [ ]:
# Embed a single query and inspect the vector
sample_query = "How many PTO days does a new employee receive?"
query_vector = embed_query(sample_query, embed_model)

print(f"Query      : '{sample_query}'")
print(f"Vector dim : {len(query_vector)}")
print(f"First 8    : {[round(v, 4) for v in query_vector[:8]]}")
print(f"L2 norm    : {sum(v**2 for v in query_vector)**0.5:.4f}")


In [ ]:
# Embed a small batch of chunks and measure cosine similarity
import math

def cosine_similarity(a, b):
    dot = sum(x*y for x, y in zip(a, b))
    mag_a = math.sqrt(sum(x**2 for x in a))
    mag_b = math.sqrt(sum(x**2 for x in b))
    return dot / (mag_a * mag_b) if mag_a and mag_b else 0

sample_texts = [c.get_content() for c in chunks_default[:5]]
chunk_vectors = embed_texts(sample_texts, embed_model)

print(f"Query: '{sample_query}'\n")
for i, (text, vec) in enumerate(zip(sample_texts, chunk_vectors)):
    sim = cosine_similarity(query_vector, vec)
    print(f"Chunk {i+1} | similarity: {sim:.4f}")
    print(f"  {text[:120].replace(chr(10), ' ')}...")
    print()


## 4. Vector Store — Chroma

The pipeline uses Chroma as a local persistent vector store. On first run it
builds the index; subsequent runs load from disk (fast warm start).

`FORCE_REBUILD=true` in `.env` triggers a full rebuild.


In [ ]:
from src.vectorstore.store import get_vector_store

# Build or load the Chroma index
index = get_vector_store(
    documents=docs,
    embed_model=embed_model,
    persist_dir="../data/chroma",
)
print("Vector store ready")
print(f"  Type    : {type(index).__name__}")


In [ ]:
# Inspect the underlying Chroma collection
chroma_client = index._vector_store._collection
count = chroma_client.count()
print(f"Documents in Chroma: {count}")
print()

# Peek at stored metadata
sample = chroma_client.peek(limit=3)
for i, (doc_id, meta, text) in enumerate(
    zip(sample["ids"], sample["metadatas"], sample["documents"])
):
    print(f"[{i+1}] ID   : {doc_id}")
    print(f"     Meta : {meta}")
    print(f"     Text : {text[:100]}...")
    print()


## 5. Retrieval

We compare retrieval quality across different configurations:
- `k` (number of results): 1 vs 3 vs 5
- Similarity score threshold: strict (0.7) vs relaxed (0.3)
- Query reformulation effect


In [ ]:
from src.retrieval.retriever import retrieve_with_scores, format_retrieved_context

queries = [
    "How many PTO days does a new employee receive?",
    "What is the parental leave policy?",
    "When are bonuses paid out?",
    "What happens if I get a rating of 2?",
]

print("=== Retrieval Results (k=3, cutoff=0.5) ===\n")
for query in queries:
    results = retrieve_with_scores(index, query, similarity_top_k=3)
    print(f"Query: '{query}'")
    for i, (node, score) in enumerate(results, 1):
        print(f"  [{i}] score={score:.4f} | {node.get_content()[:100].replace(chr(10), ' ')}...")
    print()


In [ ]:
# Compare k=1 vs k=5 on a harder query
hard_query = "Does equity continue vesting while on parental leave?"

print(f"Query: '{hard_query}'\n")
for k in [1, 3, 5]:
    results = retrieve_with_scores(index, hard_query, similarity_top_k=k)
    print(f"k={k}: {len(results)} result(s)")
    for node, score in results:
        print(f"  score={score:.4f} | {node.get_content()[:120].replace(chr(10), ' ')}...")
    print()


In [ ]:
# Show formatted context string (what goes into the LLM prompt)
from src.retrieval.retriever import retrieve_with_metadata

result = retrieve_with_metadata(index, "What is the 401k match?")
print(f"Success     : {result['success']}")
print(f"Num results : {result['num_results']}")
print(f"Max score   : {result['max_score']:.4f}")
print()
print("--- Formatted context ---")
print(result['context'])


## 6. End-to-End Generation

We run the full RAG pipeline — retrieve context, inject into prompt,
generate answer — and compare it against the model's answer without retrieval.


In [ ]:
from src.generation.generator import get_llm, generate

llm = get_llm()
print(f"LLM provider : {os.getenv('LLM_PROVIDER', 'openai')}")
print(f"LLM model    : {os.getenv('LLM_MODEL', 'gpt-4o')}")


In [ ]:
test_questions = [
    "How many PTO days does a new employee receive in their first year?",
    "What is the 401k company match percentage?",
    "What is the RSU vesting schedule for VP-level employees?",
]

for question in test_questions:
    result = retrieve_with_metadata(index, question)
    answer = generate(llm, question, result['context'])
    print(f"Q: {question}")
    print(f"A: {answer}")
    print(f"   (sources: {result['num_results']}, max_score: {result['max_score']:.4f})")
    print()


In [ ]:
# RAG vs no-RAG comparison
comparison_q = "Does equity vesting continue during parental leave?"

# Without RAG
no_rag_answer = llm.complete(comparison_q).text

# With RAG
result = retrieve_with_metadata(index, comparison_q)
rag_answer = generate(llm, comparison_q, result['context'])

print("Question:", comparison_q)
print()
print("WITHOUT RAG:")
print(no_rag_answer)
print()
print("WITH RAG:")
print(rag_answer)


## 7. Evaluation on Ground Truth

We run a subset of the 35 ground truth Q&A pairs through the pipeline and
score answers using LLM-as-judge (faithfulness, relevance, correctness).


In [ ]:
import json

with open("../evaluation/qa_pairs.json") as f:
    eval_data = json.load(f)

pairs = eval_data["pairs"]
print(f"Total Q&A pairs: {eval_data['total']}")

from collections import Counter
diff_counts = Counter(p["difficulty"] for p in pairs)
for d in ["Easy", "Medium", "Hard"]:
    print(f"  {d}: {diff_counts.get(d, 0)}")


In [ ]:
# Run evaluation on a sample (3 easy, 2 medium, 1 hard)
from collections import defaultdict

sample = (
    [p for p in pairs if p["difficulty"] == "Easy"][:3] +
    [p for p in pairs if p["difficulty"] == "Medium"][:2] +
    [p for p in pairs if p["difficulty"] == "Hard"][:1]
)

JUDGE_PROMPT = """You are evaluating a RAG system answer against a ground truth.

Question: {question}
Ground Truth: {expected}
System Answer: {answer}

Score each dimension 1-5:
- Correctness: Does the answer match the ground truth?
- Faithfulness: Is the answer grounded in the retrieved context?
- Relevance: Does the answer address the question?

Respond ONLY with JSON: {{"correctness": X, "faithfulness": X, "relevance": X}}
"""

results = []
for p in sample:
    ret = retrieve_with_metadata(index, p["question"])
    answer = generate(llm, p["question"], ret["context"])

    judge_input = JUDGE_PROMPT.format(
        question=p["question"],
        expected=p["expected_answer"],
        answer=answer
    )
    try:
        raw = llm.complete(judge_input).text
        import re
        match = re.search(r'\{.*?\}', raw, re.DOTALL)
        scores = json.loads(match.group()) if match else {}
    except Exception:
        scores = {}

    results.append({
        "id": p["id"],
        "difficulty": p["difficulty"],
        "question": p["question"],
        "expected": p["expected_answer"],
        "answer": answer,
        "scores": scores,
        "max_score": ret["max_score"],
    })

print(f"Evaluated {len(results)} questions\n")
for r in results:
    s = r["scores"]
    print(f"[{r['id']}] {r['difficulty']}")
    print(f"  Q  : {r['question']}")
    print(f"  A  : {r['answer'][:100]}...")
    print(f"  Scores : correctness={s.get('correctness','?')} faithfulness={s.get('faithfulness','?')} relevance={s.get('relevance','?')}")
    print(f"  Retrieval score: {r['max_score']:.4f}")
    print()


In [ ]:
# Aggregate scores
from statistics import mean

if results:
    corr = [r["scores"].get("correctness", 0) for r in results if r["scores"]]
    faith = [r["scores"].get("faithfulness", 0) for r in results if r["scores"]]
    rel = [r["scores"].get("relevance", 0) for r in results if r["scores"]]

    print("=== Evaluation Summary ===")
    print(f"Questions evaluated : {len(results)}")
    if corr:
        print(f"Avg correctness     : {mean(corr):.2f} / 5")
        print(f"Avg faithfulness    : {mean(faith):.2f} / 5")
        print(f"Avg relevance       : {mean(rel):.2f} / 5")

    print()
    print("By difficulty:")
    for diff in ["Easy", "Medium", "Hard"]:
        subset = [r for r in results if r["difficulty"] == diff and r["scores"]]
        if subset:
            avg = mean(r["scores"].get("correctness", 0) for r in subset)
            print(f"  {diff}: avg correctness {avg:.2f} / 5 ({len(subset)} questions)")


## 8. Next Steps

**Run the full evaluation suite:**
```bash
python evaluation/eval.py --qa evaluation/qa_pairs.json --output evaluation/results.json
```

**Launch the Gradio demo UI:**
```bash
gradio ui/app.py
```

**Run the pipeline in interactive CLI mode:**
```bash
python src/pipeline.py
```

**Explore the agentic version** (LangChain + LangGraph + Pinecone) in the
companion repo: [agentic-rag](https://github.com/tohio/agentic-rag)

---
*Meridian Capital Group HR Assistant — rag-pipeline exploration notebook*
